In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: engineering-state
import json
from collections import Counter
from pathlib import Path
import pandas as pd

ROOT = Path("..")
queue_path = ROOT / "refine-logs/queue/queue_state.json"
audit_path = ROOT / "results/checkpoint_store/compatibility_audit.json"
catalog_path = ROOT / "results/checkpoint_store/catalog.jsonl"

state = json.loads(queue_path.read_text())
audit = json.loads(audit_path.read_text())
queue_counts = Counter(j.get("status", "unknown") for j in state["jobs"])
catalog_rows = [json.loads(line) for line in catalog_path.read_text().splitlines() if line.strip()]
catalog_counts = Counter(r.get("status", "unknown") for r in catalog_rows)

pd.DataFrame({
    "layer": ["queue"] * len(queue_counts) + ["checkpoint catalog"] * len(catalog_counts),
    "state": list(queue_counts) + list(catalog_counts),
    "count": list(queue_counts.values()) + list(catalog_counts.values())
})

,layer,state,count
0,queue,completed,11
1,queue,running,1
2,queue,pending,468
3,checkpoint catalog,remote_verified,10
4,checkpoint catalog,error,135
5,checkpoint catalog,cached,1


In [3]:
#| label: engineering-contract
contract = audit["contract"]
pd.DataFrame({
    "field": [
        "contract id", "source bundle SHA-256", "state schema SHA-256",
        "sample rate", "units", "normalization",
        "train records", "validation records", "test records"
    ],
    "value": [
        contract["contract_id"], contract["approved_source_bundle_sha256"],
        contract["state_schema_sha256"], contract["preprocessing"]["sample_rate_hz"],
        contract["preprocessing"]["units"], contract["preprocessing"]["normalization"],
        contract["split_content_roots"]["train"]["records"],
        contract["split_content_roots"]["val"]["records"],
        contract["split_content_roots"]["test"]["records"]
    ]
})

,field,value
0,contract id,factorial-v4-content-pinned-20260731
1,source bundle SHA-256,6e262086df8e995d90cea19208f189d78658959641069e...
2,state schema SHA-256,6d350ee6785b98efe72e8e058cb717fd5a40201289e156...
3,sample rate,500
4,units,mV
5,normalization,none
6,train records,17418
7,validation records,2183
8,test records,2198


In [4]:
#| label: compatible-inference-readiness
#| tbl-cap: Exact archived-model inference readiness on one real PTB-XL test record. Timings are operational observations, not model-quality endpoints.
import hashlib

inference_root = ROOT / "results/factorial_mixed_level/inference_readiness"
inference_summary_path = inference_root / "summary.json"
inference_csv_path = inference_root / "per_model_inference_readiness.csv"
inference_lead_csv_path = inference_root / "per_model_per_lead_case_metrics.csv"
inference_code_path = ROOT / "scripts/benchmark_factorial_inference_readiness.py"
inference_summary = json.loads(inference_summary_path.read_text())
inference_csv_bytes = inference_csv_path.read_bytes()
inference_lead_csv_bytes = inference_lead_csv_path.read_bytes()

assert hashlib.sha256(inference_csv_bytes).hexdigest() == inference_summary["csv_sha256"]
assert hashlib.sha256(inference_lead_csv_bytes).hexdigest() == inference_summary["per_lead_case_metrics_csv_sha256"]
assert hashlib.sha256(audit_path.read_bytes()).hexdigest() == inference_summary["compatibility_audit_sha256"]
assert hashlib.sha256(inference_code_path.read_bytes()).hexdigest() == inference_summary["benchmark_code_sha256"]

inference_rows = pd.read_csv(inference_csv_path)
inference_lead_rows = pd.read_csv(inference_lead_csv_path)
assert len(inference_rows) == audit["counts"]["compatible"]
assert inference_summary["models_completed"] == len(inference_rows)
assert inference_rows.finite.all() and inference_summary["all_finite"]
assert inference_summary["cache_retained_bytes"] == 0
assert len(inference_lead_rows) == len(inference_rows) * 9
assert inference_lead_rows[["mse", "mae", "pearson", "variance_ratio"]].notna().all().all()

display(pd.DataFrame([
    ("Compatible identities expected", inference_summary["models_expected"]),
    ("Models strictly loaded and executed", inference_summary["models_completed"]),
    ("Finite reconstructions", int(inference_rows.finite.sum())),
    ("Input tensor SHA-256", inference_summary["input_sha256"]),
    ("Prepared input shape", str(inference_summary["prepared_input_shape"])),
    ("Reconstruction shape", str(inference_summary["output_shape"])),
    ("Logical checkpoint bytes traversed", inference_summary["checkpoint_logical_bytes"]),
    ("Checkpoint-cache bytes retained", inference_summary["cache_retained_bytes"]),
], columns=["Readiness gate", "Observed value"]))

display(inference_rows[[
    "model_id", "checkpoint_sha256", "load_and_materialize_seconds",
    "forward_median_seconds", "repeats", "output_mean", "output_std", "finite",
]])

,Readiness gate,Observed value
0,Compatible identities expected,11
1,Models strictly loaded and executed,11
2,Finite reconstructions,11
3,Input tensor SHA-256,f10cde0f89950700ad41a3289c2ef607536acdb7c723ae...
4,Prepared input shape,"[1, 3, 5024]"
5,Reconstruction shape,"[1, 12, 5000]"
6,Logical checkpoint bytes traversed,451120784
7,Checkpoint-cache bytes retained,0


,model_id,checkpoint_sha256,load_and_materialize_seconds,forward_median_seconds,repeats,output_mean,output_std,finite
0,f_1000000_s42,ea2df9939d4438de51c702ddd887edf2ccd57b8e2c0947...,0.836061,0.480534,3,0.001137,0.165322,True
1,f_1000001_s42,b5e7d69c9311da7351f6fc22b42294d8869767e34bac02...,0.812432,0.140895,3,0.001997,0.194937,True
2,f_1000002_s42,f7b35963db704af91c2d3145c8b5870369d9aa5fd43ce9...,0.722654,0.142742,3,0.002002,0.180973,True
3,f_1000003_s42,0d6d1620f831eae098313d8be11ac738f37b202f696aa2...,0.719299,0.492561,3,0.002004,0.172353,True
4,f_1011012_s42,a55a3f42e986fee995bb7d041e8e3e8606c4af63e9c48c...,0.772386,0.150425,3,-0.003926,0.196440,True
5,f_1011013_s42,5c4a6f3116f0faaab1587ebbadd9066a921147aef84be9...,0.694392,0.140348,3,-0.017461,0.189490,True
6,f_1011014_s42,2cb7851a810af55f4664ef43320d328fea868b64817555...,0.398526,0.147243,3,-0.014453,0.196221,True
7,f_1011100_s42,9463d8d935c356d740bc8f8d62b8d1abc1b244eef747ae...,1.273786,0.140745,3,-0.000361,0.201576,True
8,f_1011101_s42,c76c9f49c4ab89d41319582bd743a1ef9ff5742f6527e1...,0.656336,0.136599,3,-0.000182,0.204173,True
9,f_1011102_s42,ddca24dcf77a6914e1f0de54d6ab8ca2ea68449ad0bbf9...,1.760438,0.144688,3,-0.000321,0.201519,True


In [5]:
#| label: compatible-inference-index-case-lead-summary
#| tbl-cap: Missing-lead reconstruction diagnostics across the ten compatible models for PTB-XL test record 100 only.
lead_order = ["III", "aVR", "aVL", "aVF", "V1", "V3", "V4", "V5", "V6"]
index_case_lead_summary = (
    inference_lead_rows.groupby("lead", as_index=False)
    .agg(
        models=("model_id", "nunique"),
        median_mse=("mse", "median"),
        minimum_mse=("mse", "min"),
        maximum_mse=("mse", "max"),
        median_pearson=("pearson", "median"),
        minimum_pearson=("pearson", "min"),
        maximum_pearson=("pearson", "max"),
        median_variance_ratio=("variance_ratio", "median"),
    )
    .set_index("lead")
    .loc[lead_order]
    .reset_index()
)
display(index_case_lead_summary)

,lead,models,median_mse,minimum_mse,maximum_mse,median_pearson,minimum_pearson,maximum_pearson,median_variance_ratio
0,III,11,0.009262,0.003692,0.017923,0.609622,0.328458,0.800903,1.182602
1,aVR,11,0.010037,0.004230,0.014290,0.740987,0.601232,0.924491,1.181895
2,aVL,11,0.016028,0.010720,0.019679,0.461960,-0.278169,0.732835,1.652092
3,aVF,11,0.006791,0.005124,0.022792,0.741614,0.564542,0.755659,1.493677
4,V1,11,0.047100,0.018412,0.115758,-0.114377,-0.683032,0.833943,0.921791
5,V3,11,0.045895,0.030265,0.064754,0.876531,0.792287,0.892733,3.410634
6,V4,11,0.018577,0.017091,0.050870,0.864180,0.746001,0.909019,2.497810
7,V5,11,0.010605,0.008674,0.021743,0.836450,0.794302,0.897521,1.444931
8,V6,11,0.018581,0.007800,0.037545,0.669951,0.604340,0.802553,1.290576


In [6]:
#| label: compatible-inference-index-case-heatmap
#| fig-cap: Per-model, per-missing-lead MSE on one indexed PTB-XL test record. This heatmap diagnoses output behavior and must not be read as cohort-level model ranking.
import plotly.express as px

case_mse = (
    inference_lead_rows.pivot(index="model_id", columns="lead", values="mse")
    .reindex(columns=lead_order)
    .sort_index()
)
case_heatmap = px.imshow(
    case_mse,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Reconstructed missing lead", "y": "Exact model identity", "color": "MSE"},
)
case_heatmap.update_layout(height=520)
case_heatmap.show()

In [7]:
#| label: compatible-model-table
model_rows = []
for m in audit["models"]:
    model_rows.append({
        "model_id": m.get("model_id"),
        "compatible": bool(m.get("compatible")),
        "reason": "; ".join(m.get("reasons", [])) or "approved",
        "checkpoint_sha256": str(m.get("checkpoint_sha256", ""))[:16]
    })
pd.DataFrame(model_rows).sort_values(
    ["compatible", "model_id"], ascending=[False, True]
).head(25)

,model_id,compatible,reason,checkpoint_sha256
0,f_1000000_s42,True,approved,ea2df9939d4438de
1,f_1000001_s42,True,approved,b5e7d69c9311da73
2,f_1000002_s42,True,approved,f7b35963db704af9
3,f_1000003_s42,True,approved,0d6d1620f831eae0
4,f_1011012_s42,True,approved,a55a3f42e986fee9
5,f_1011013_s42,True,approved,5c4a6f3116f0faaa
6,f_1011014_s42,True,approved,2cb7851a810af55f
7,f_1011100_s42,True,approved,9463d8d935c356d7
8,f_1011101_s42,True,approved,c76c9f49c4ab89d4
9,f_1011102_s42,True,approved,ddca24dcf77a6914


In [8]:
#| label: log-coverage
import re

log_dir = ROOT / "refine-logs/queue/logs"
logs = sorted(log_dir.glob("f_*_s*.log"))
rows = []
for path in logs:
    content = path.read_text(errors="replace")
    epochs = [int(x) for x in re.findall(r"Epoch\s+(\d+)", content)]
    rows.append({
        "model_id": path.stem,
        "bytes": path.stat().st_size,
        "last_epoch_seen": max(epochs) if epochs else None,
        "cuda_oom_mentions": content.lower().count("out of memory"),
        "traceback_mentions": content.count("Traceback")
    })
log_df = pd.DataFrame(rows)
pd.DataFrame({
    "quantity": ["log files", "logs with epoch markers", "OOM mentions", "tracebacks"],
    "value": [len(log_df), log_df.last_epoch_seen.notna().sum() if len(log_df) else 0,
              int(log_df.cuda_oom_mentions.sum()) if len(log_df) else 0,
              int(log_df.traceback_mentions.sum()) if len(log_df) else 0]
})

,quantity,value
0,log files,247
1,logs with epoch markers,18
2,OOM mentions,0
3,tracebacks,235


In [9]:
#| label: compatible-training-summary
#| tbl-cap: Optimization diagnostics for current-contract compatible checkpoints.
diagnostic_root = ROOT / "results/factorial_mixed_level/training_diagnostics"
epoch_curves = pd.read_csv(diagnostic_root / "compatible_epoch_curves.csv")
model_diagnostics = pd.read_csv(
    diagnostic_root / "compatible_model_summary.csv"
)
training_diagnostic_status = json.loads(
    (diagnostic_root / "summary.json").read_text()
)
operational_eta = pd.read_csv(
    diagnostic_root / "operational_eta_by_kernel.csv"
)

model_diagnostics.assign(
    duration_minutes=model_diagnostics.duration_seconds / 60,
    val_mse_reduction_percent=-100 * model_diagnostics.val_mse_relative_change,
)[[
    "model_id", "mmd_kernel", "epochs", "duration_minutes",
    "first_val_mse", "last_val_mse", "val_mse_reduction_percent",
    "cuda_oom_mentions", "traceback_mentions", "nonfinite_mentions",
]]

,model_id,mmd_kernel,epochs,duration_minutes,first_val_mse,last_val_mse,val_mse_reduction_percent,cuda_oom_mentions,traceback_mentions,nonfinite_mentions
0,f_1000000_s42,0,10,20.074931,0.0791,0.0305,61.441214,0,0,0
1,f_1000001_s42,1,10,24.079657,0.1077,0.0384,64.345404,0,0,0
2,f_1000002_s42,2,10,23.081824,0.1071,0.0342,68.067227,0,0,0
3,f_1000003_s42,3,10,26.082670,0.0973,0.0310,68.139774,0,0,0
4,f_1011012_s42,2,10,21.111653,0.1791,0.0504,71.859296,0,0,0
5,f_1011013_s42,3,10,18.077916,0.2017,0.0536,73.425880,0,0,0
6,f_1011014_s42,4,10,23.078487,0.1983,0.0540,72.768533,0,0,0
7,f_1011100_s42,0,10,17.067856,0.1351,0.0520,61.509993,0,0,0
8,f_1011101_s42,1,10,18.066144,0.1323,0.0522,60.544218,0,0,0
9,f_1011102_s42,2,10,26.093416,0.1351,0.0521,61.435973,0,0,0


In [10]:
#| label: compatible-operational-eta
#| tbl-cap: Single-GPU operational ETA from current-contract run durations and the live remaining kernel mix.
eta = training_diagnostic_status["operational_eta"]
display(pd.DataFrame([
    ("Compatible duration samples", training_diagnostic_status["compatible_models"]),
    ("Pending or running jobs", eta["remaining_pending_or_running_jobs"]),
    ("Estimated remaining hours", eta["estimated_remaining_hours"]),
    ("Estimated remaining days", eta["estimated_remaining_days"]),
    ("Estimated completion (UTC)", eta["estimated_completion_utc"]),
    ("Observed-min scenario (hours)", eta["observed_min_scenario_hours"]),
    ("Observed-max scenario (hours)", eta["observed_max_scenario_hours"]),
], columns=["ETA field", "Value"]))

display(operational_eta[[
    "mmd_kernel", "compatible_duration_samples", "observed_min_minutes",
    "observed_median_minutes", "observed_max_minutes", "pending_jobs",
    "running_jobs", "running_elapsed_minutes", "estimated_remaining_minutes",
]])

,ETA field,Value
0,Compatible duration samples,11
1,Pending or running jobs,469
2,Estimated remaining hours,166.824306
3,Estimated remaining days,6.951013
4,Estimated completion (UTC),2026-08-08T06:00:38.582606+00:00
5,Observed-min scenario (hours),152.258798
6,Observed-max scenario (hours),186.520155


,mmd_kernel,compatible_duration_samples,observed_min_minutes,observed_median_minutes,observed_max_minutes,pending_jobs,running_jobs,running_elapsed_minutes,estimated_remaining_minutes
0,0,2,17.067856,18.571394,20.074931,94,0,0.000000,1745.710991
1,1,2,18.066144,21.072900,24.079657,94,0,0.000000,1980.852610
2,2,3,21.111653,23.081824,26.093416,93,0,0.000000,2146.609612
3,3,3,18.077916,20.946056,26.082670,93,0,0.000000,1947.983168
4,4,1,23.078487,23.078487,23.078487,94,1,4.154306,2188.301993


In [11]:
#| label: compatible-validation-mse-curves
#| fig-cap: Validation MSE trajectories from current-contract compatible runs. Total composite loss is not plotted because its scale changes with active loss terms.
import plotly.express as px

fig = px.line(
    epoch_curves,
    x="epoch",
    y="val_mse",
    color="model_id",
    markers=True,
    labels={"val_mse": "Validation MSE", "epoch": "Epoch", "model_id": "Model ID"},
)
fig.update_layout(height=480, legend_title_text="Exact model identity")
fig.show()

In [12]:
#| label: temporal-morphology-acceptance-gate
#| tbl-cap: Live acceptance state for generation-bound morphology artifacts. Partial accepted models are operational diagnostics, not a factorial comparison.
temporal_root = ROOT / "results/factorial_v4/temporal_mmd_generation_bound"
temporal_status = json.loads((temporal_root / "accepted_summary.json").read_text())
temporal_models = pd.read_csv(temporal_root / "accepted_model_artifacts.csv")
temporal_features = pd.read_csv(temporal_root / "accepted_feature_summary.csv")
temporal_exclusions = pd.read_csv(temporal_root / "excluded_artifacts.csv")

pd.DataFrame([
    ("Current compatible checkpoints", temporal_status["eligible_compatible_models"]),
    ("Accepted evaluated checkpoints", temporal_status["accepted_models"]),
    ("Accepted feature rows", temporal_status["accepted_feature_rows"]),
    ("Excluded stale/invalid artifacts", temporal_status["excluded_artifacts"]),
    ("Complete for factorial inference", temporal_status["complete_for_factorial_inference"]),
    ("Evaluator SHA-256", temporal_status["evaluation_code_sha256"]),
    ("Target-cache SHA-256", temporal_status["target_feature_cache_sha256"]),
], columns=["Gate field", "Value"])

,Gate field,Value
0,Current compatible checkpoints,11
1,Accepted evaluated checkpoints,3
2,Accepted feature rows,36
3,Excluded stale/invalid artifacts,0
4,Complete for factorial inference,False
5,Evaluator SHA-256,7f5323ecdfcf6601d5972d804864498db55425f86f4d05...
6,Target-cache SHA-256,057765b02da864d334dd7a65c5440b96126ad4354986c2...


In [13]:
#| label: temporal-morphology-exclusions
#| tbl-cap: Artifacts excluded from the current evaluator generation.
if temporal_exclusions.empty:
    display(pd.DataFrame({"status": ["No exclusions"]}))
else:
    display(temporal_exclusions)

,status
0,No exclusions


In [14]:
#| label: temporal-morphology-coverage
#| fig-cap: Per-model detector/pairing coverage for accepted partial artifacts. Coverage is shown to diagnose measurement failure, not to rank loss masks before grid completion.
if temporal_features.empty:
    display(pd.DataFrame({
        "status": ["No artifact yet passes the current evaluator and digest gate"]
    }))
else:
    import plotly.express as px
    coverage_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="record_pair_coverage",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "n_records_total", "n_records_real_detected",
            "n_records_recon_detected", "n_records_paired", "n_beats",
        ],
        labels={
            "clinical_feature": "Feature",
            "record_pair_coverage": "Records with a finite paired measurement",
        },
    )
    coverage_plot.update_yaxes(range=[0, 1])
    coverage_plot.update_layout(height=500, legend_title_text="Exact model identity")
    coverage_plot.show()

In [15]:
#| label: temporal-morphology-partial-effect-diagnostics
#| fig-cap: Variance ratios for accepted partial artifacts. The horizontal line marks equal reconstructed/target variance; these are feature-detector diagnostics, not completed factorial effects.
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted feature rows"]}))
else:
    variance_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="variance_ratio",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "mean_real", "mean_recon", "ba_robust_slope", "record_pair_coverage",
        ],
        labels={
            "clinical_feature": "Feature",
            "variance_ratio": "Detected reconstructed / target variance",
        },
    )
    variance_plot.add_hline(y=1.0, line_dash="dash", line_color="black")
    variance_plot.update_layout(height=500, legend_title_text="Exact model identity")
    variance_plot.show()

    partial_diagnostic = pd.DataFrame([
        ("Accepted models", temporal_features.model_id.nunique()),
        ("Lead-feature rows", len(temporal_features)),
        ("Rows with variance ratio < 1", int((temporal_features.variance_ratio < 1).sum())),
        ("Variance-ratio range", f"{temporal_features.variance_ratio.min():.4f}–{temporal_features.variance_ratio.max():.4f}"),
        ("Pair-coverage range", f"{temporal_features.record_pair_coverage.min():.2%}–{temporal_features.record_pair_coverage.max():.2%}"),
    ], columns=["Partial diagnostic", "Observed value"])
    display(partial_diagnostic)

    coverage_failures = temporal_features.assign(
        records_without_pair=(
            temporal_features.n_records_total - temporal_features.n_records_paired
        )
    )[[
        "model_id", "lead", "clinical_feature", "n_records_total",
        "n_records_real_detected", "n_records_recon_detected",
        "n_records_paired", "records_without_pair", "record_pair_coverage",
    ]].sort_values("record_pair_coverage")
    display(coverage_failures)

,Partial diagnostic,Observed value
0,Accepted models,3
1,Lead-feature rows,36
2,Rows with variance ratio < 1,31
3,Variance-ratio range,0.0577–3.2238
4,Pair-coverage range,89.26%–99.95%


,model_id,lead,clinical_feature,n_records_total,n_records_real_detected,n_records_recon_detected,n_records_paired,records_without_pair,record_pair_coverage
35,f_1011013_s42,V6,QT_Interval_ms,2198,2119,2024,1962,236,0.892630
23,f_1011012_s42,V6,QT_Interval_ms,2198,2119,2066,2001,197,0.910373
29,f_1011013_s42,V3,QT_Interval_ms,2198,2094,2113,2042,156,0.929026
17,f_1011012_s42,V3,QT_Interval_ms,2198,2094,2125,2055,143,0.934941
31,f_1011013_s42,V6,Q_Amp,2198,2167,2094,2068,130,0.940855
5,f_1000000_s42,V3,QT_Interval_ms,2198,2094,2157,2078,120,0.945405
11,f_1000000_s42,V6,QT_Interval_ms,2198,2119,2150,2082,116,0.947225
19,f_1011012_s42,V6,Q_Amp,2198,2167,2120,2091,107,0.951319
16,f_1011012_s42,V3,T_Amp,2198,2126,2151,2103,95,0.956779
28,f_1011013_s42,V3,T_Amp,2198,2126,2154,2104,94,0.957234


In [16]:
#| label: temporal-morphology-model-snapshot
#| tbl-cap: Descriptive model-level morphology snapshot for accepted artifacts. Masks differ in multiple loss factors, so this table is not a causal contrast.
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted model snapshots"]}))
else:
    model_snapshot = (
        temporal_features.groupby(["model_id", "model_mask", "mmd_kernel"], as_index=False)
        .agg(
            lead_feature_rows=("clinical_feature", "size"),
            minimum_pair_coverage=("record_pair_coverage", "min"),
            maximum_pair_coverage=("record_pair_coverage", "max"),
            median_variance_ratio=("variance_ratio", "median"),
            rows_below_unit_variance=("variance_ratio", lambda x: int((x < 1).sum())),
            maximum_variance_ratio=("variance_ratio", "max"),
        )
    )
    display(model_snapshot)

,model_id,model_mask,mmd_kernel,lead_feature_rows,minimum_pair_coverage,maximum_pair_coverage,median_variance_ratio,rows_below_unit_variance,maximum_variance_ratio
0,f_1000000_s42,1000000,0,12,0.945405,0.999545,0.255172,11,1.517615
1,f_1011012_s42,1011012,2,12,0.910373,0.999545,0.288437,10,3.130165
2,f_1011013_s42,1011013,3,12,0.892630,0.999545,0.128988,10,3.223795


In [17]:
#| label: temporal-morphology-kernel-2-vs-3
#| tbl-cap: Same-mask, same-seed descriptive contrast between MMD kernels 2 and 3. Aggregate detector-paired features are shown without inferential uncertainty.
matched_ids = {"f_1011012_s42", "f_1011013_s42"}
if matched_ids.issubset(set(temporal_features.model_id)):
    matched = temporal_features[temporal_features.model_id.isin(matched_ids)][[
        "model_id", "lead", "clinical_feature", "variance_ratio",
        "record_pair_coverage", "mean_recon",
    ]]
    kernel_2 = matched[matched.model_id.eq("f_1011012_s42")].drop(columns="model_id")
    kernel_3 = matched[matched.model_id.eq("f_1011013_s42")].drop(columns="model_id")
    kernel_contrast = kernel_2.merge(
        kernel_3,
        on=["lead", "clinical_feature"],
        suffixes=("_kernel2", "_kernel3"),
        validate="one_to_one",
    )
    kernel_contrast["variance_ratio_delta_kernel3_minus_kernel2"] = (
        kernel_contrast.variance_ratio_kernel3 - kernel_contrast.variance_ratio_kernel2
    )
    kernel_contrast["coverage_delta_kernel3_minus_kernel2"] = (
        kernel_contrast.record_pair_coverage_kernel3
        - kernel_contrast.record_pair_coverage_kernel2
    )
    display(kernel_contrast)
    display(pd.DataFrame([
        ("Lead-feature rows", len(kernel_contrast)),
        ("Rows with lower variance ratio under kernel 3", int((kernel_contrast.variance_ratio_delta_kernel3_minus_kernel2 < 0).sum())),
        ("Median variance-ratio delta, kernel 3 − kernel 2", kernel_contrast.variance_ratio_delta_kernel3_minus_kernel2.median()),
        ("Median pairing-coverage delta, kernel 3 − kernel 2", kernel_contrast.coverage_delta_kernel3_minus_kernel2.median()),
    ], columns=["Descriptive contrast", "Observed value"]))
else:
    display(pd.DataFrame({"status": ["Matched kernel-2/kernel-3 artifacts are not both accepted"]}))

,lead,clinical_feature,variance_ratio_kernel2,record_pair_coverage_kernel2,mean_recon_kernel2,variance_ratio_kernel3,record_pair_coverage_kernel3,mean_recon_kernel3,variance_ratio_delta_kernel3_minus_kernel2,coverage_delta_kernel3_minus_kernel2
0,V3,P_Amp,0.266592,0.988171,-0.094828,0.116939,0.989991,-0.209050,-0.149652,0.001820
1,V3,Q_Amp,0.152321,0.984076,-0.184168,0.088251,0.977707,-0.269601,-0.064070,-0.006369
2,V3,R_Amp,0.107571,0.992721,0.559152,0.061381,0.993176,0.366403,-0.046190,0.000455
3,V3,S_Amp,0.119504,0.992266,-1.189953,0.057737,0.992721,-1.064996,-0.061766,0.000455
4,V3,T_Amp,0.310282,0.956779,0.190795,0.141036,0.957234,-0.035151,-0.169245,0.000455
5,V3,QT_Interval_ms,1.133082,0.934941,598.184737,2.157757,0.929026,701.528623,1.024675,-0.005914
6,V6,P_Amp,0.333661,0.997270,-0.071277,0.275398,0.998635,-0.115187,-0.058263,0.001365
7,V6,Q_Amp,0.117223,0.951319,-0.173167,0.093460,0.940855,-0.210498,-0.023763,-0.010464
8,V6,R_Amp,0.332117,0.999545,0.903072,0.370068,0.999545,0.977961,0.037951,0.000000
9,V6,S_Amp,0.123287,0.999545,-0.188337,0.111928,0.999545,-0.219176,-0.011358,0.000000


,Descriptive contrast,Observed value
0,Lead-feature rows,12.000000
1,Rows with lower variance ratio under kernel 3,9.000000
2,"Median variance-ratio delta, kernel 3 − kernel 2",-0.052227
3,"Median pairing-coverage delta, kernel 3 − kern...",0.000000
